# Aula 11 — BERT para classificação de texto

**Do embedding contextual ao fine-tuning supervisionado**

Na Aula 10, usamos um Transformer pré-treinado para observar representações contextuais.

Agora vamos dar o próximo passo:

> adaptar um modelo pré-treinado para classificar textos em categorias conhecidas.

Esse processo é chamado de **fine-tuning**.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que é fine-tuning;
- distinguir pré-treinamento de ajuste supervisionado;
- entender o papel do tokenizer em modelos BERT-like;
- preparar textos e rótulos para classificação;
- usar um modelo pré-treinado para classificação de sequência;
- executar um pequeno treino supervisionado;
- avaliar o resultado com métricas básicas;
- comparar conceitualmente BERT com o baseline TF-IDF + Naive Bayes.


## 📘 Glossário da aula

Conceitos centrais: **fine-tuning · pré-treinamento · tokenizer · sequence classification · epoch · batch · learning rate · transfer learning**.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 🧩 Modelo pré-treinado anexado ao notebook

Nesta aula, o modelo será usado como um **Kaggle Model anexado e versionado**.

Isso permite executar o notebook com **Internet OFF**, mantendo a dependência explícita dentro do ambiente Kaggle.

Recurso homologado para esta aula:

```text
goddiao/distilbert-base-multilingual-cased
framework: PyTorch
variation: default
version: 1
```

Caminho esperado após anexar o modelo:

```text
/kaggle/input/models/goddiao/distilbert-base-multilingual-cased/pytorch/default/1/distilbert-base-multilingual-cased
```

> Antes de executar, use **Add Models** no Kaggle e anexe exatamente esse recurso. A internet deve permanecer desabilitada.


## 2. Pré-treinamento versus fine-tuning

Um modelo como BERT é inicialmente treinado em grandes coleções de texto para aprender regularidades gerais da linguagem.

Depois, podemos adaptar esse conhecimento para uma tarefa específica.

```text
pré-treinamento
→ conhecimento linguístico geral

fine-tuning
→ adaptação para uma tarefa específica
```

Nesta aula, nossa tarefa será classificar mensagens como `duvida`, `reclamacao` ou `elogio`.


## 3. Dataset didático

Vamos usar um conjunto pequeno e balanceado apenas para observar o pipeline completo.


In [ ]:
MODEL_NAME = "distilbert-base-multilingual-cased"
MODEL_DIR = "/kaggle/input/models/goddiao/distilbert-base-multilingual-cased/pytorch/default/1/distilbert-base-multilingual-cased"

print("Modelo planejado:", MODEL_NAME)
print("Diretório local:", MODEL_DIR)
print("Objetivo: fazer fine-tuning supervisionado usando o modelo Kaggle anexado.")


In [ ]:
texts = [
    "como altero minha senha",
    "onde vejo minha fatura",
    "posso pagar amanhã",
    "como atualizo meu cadastro",
    "qual o prazo para resposta",
    "como cancelo o serviço",
    "meu pedido não chegou",
    "o atendimento foi péssimo",
    "estou insatisfeito com o serviço",
    "a entrega atrasou novamente",
    "o suporte não resolveu meu problema",
    "estou muito irritado com o atendimento",
    "o atendimento foi excelente",
    "fui muito bem atendido",
    "serviço rápido e eficiente",
    "estou satisfeito com o atendimento",
    "a equipe resolveu tudo rapidamente",
    "gostei muito do suporte",
]

labels_text = [
    "duvida", "duvida", "duvida", "duvida", "duvida", "duvida",
    "reclamacao", "reclamacao", "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "elogio", "elogio", "elogio", "elogio", "elogio", "elogio",
]

label2id = {"duvida": 0, "reclamacao": 1, "elogio": 2}
id2label = {value: key for key, value in label2id.items()}
labels = [label2id[label] for label in labels_text]

print("Documentos:", len(texts))
print("Classes:", label2id)


## 4. Separando treino e validação

Como nosso dataset é muito pequeno, esta divisão serve apenas para demonstrar o fluxo.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels,
)

print("Treino:", len(X_train))
print("Validação:", len(X_val))


## 5. Carregando tokenizer e modelo

Usaremos `distilbert-base-multilingual-cased`, um modelo multilíngue compatível com português.

O tokenizer transforma texto em IDs de tokens; o modelo recebe esses IDs e aprende a associar as representações contextuais às classes.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)

print("Modelo Kaggle carregado localmente.")
print("Tokenizer:", type(tokenizer).__name__)
print("Model:", type(model).__name__)
print("Model type:", model.config.model_type)
print("Vocab size:", model.config.vocab_size)
print("Num labels:", model.config.num_labels)


### O que acontece nessa etapa?

O Kaggle fornece localmente os pesos do **encoder DistilBERT pré-treinado**.

Como o checkpoint-base não contém uma cabeça específica para nossas três classes, `AutoModelForSequenceClassification` cria uma nova `classification head`.

É esperado que o carregamento informe parâmetros `MISSING` para `pre_classifier` e `classifier`: esses pesos são inicializados para a nossa tarefa e serão aprendidos durante o fine-tuning.

```text
encoder pré-treinado
+ nova classification head
→ fine-tuning em duvida / reclamacao / elogio
```


## 6. Tokenizando os textos

Vamos truncar ou preencher sequências para um comprimento máximo curto, adequado ao nosso dataset didático.


In [ ]:
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=64,
)

val_encodings = tokenizer(
    X_val,
    truncation=True,
    padding=True,
    max_length=64,
)

print("Exemplo de input_ids:", train_encodings["input_ids"][0][:12])


## 7. Criando um Dataset PyTorch

O `Trainer` do Hugging Face espera objetos que entreguem tensores de entrada e o rótulo correspondente.


In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)

print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))


## 8. Configurando o fine-tuning

Alguns hiperparâmetros importantes aparecem agora:

- `learning_rate`: tamanho dos passos de atualização;
- `num_train_epochs`: quantas passagens completas pelo dataset;
- `per_device_train_batch_size`: quantos exemplos são processados por lote.

Manteremos tudo pequeno para fins didáticos.


In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels_eval = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_eval,
        predictions,
        average="macro",
        zero_division=0,
    )
    accuracy = accuracy_score(labels_eval, predictions)
    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

training_args = TrainingArguments(
    output_dir="./til11-output",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


## 9. Executando o fine-tuning

A próxima célula atualiza os pesos do modelo usando nosso pequeno conjunto supervisionado.

Em um projeto real, usaríamos mais dados, validação mais robusta, experimentos com hiperparâmetros e infraestrutura de treinamento adequada.


In [ ]:
trainer.train()


## 10. Avaliando o modelo

Agora medimos desempenho na partição de validação.


In [ ]:
metrics = trainer.evaluate()

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")


### Interpretação cuidadosa

Nosso dataset tem apenas 18 exemplos.

Logo, qualquer métrica obtida aqui é **didática**, não evidência de desempenho real.

O objetivo é compreender o pipeline:

```text
texto
→ tokenizer
→ Transformer pré-treinado
→ fine-tuning
→ classificação
→ avaliação
```


## 11. Testando novas mensagens

Vamos aplicar o modelo ajustado em textos novos.


In [ ]:
new_texts = [
    "não consigo acessar minha conta",
    "o atendimento foi ótimo",
    "estou muito irritado com o atraso",
]

inputs = tokenizer(
    new_texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64,
)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

predicted_ids = outputs.logits.argmax(dim=-1).tolist()

for text, predicted_id in zip(new_texts, predicted_ids):
    print(f"{id2label[predicted_id]:12} | {text}")


## 12. Comparando com nosso baseline clássico

Nas Aulas 5–7 usamos:

```text
TF-IDF + MultinomialNB
```

Agora usamos:

```text
tokenizer subword
→ Transformer pré-treinado
→ classification head
→ fine-tuning
```

O segundo pipeline é mais poderoso, mas também mais caro, mais complexo e mais dependente de infraestrutura.

Por isso, o baseline clássico continua sendo importante como referência de custo-benefício.


## 13. Transfer learning

Fine-tuning é um caso de **transfer learning**: reutilizamos conhecimento aprendido em uma tarefa ampla e transferimos esse conhecimento para uma tarefa específica.

Essa ideia foi decisiva para tornar modelos de linguagem modernos úteis em muitos problemas com menos dados rotulados do que seria necessário para treinar tudo do zero.


## 14. Exercício guiado

Use `tokenizer` e `model` já treinados.

Seu código deve:

1. classificar a mensagem `"o suporte foi excelente"`;
2. recuperar o ID previsto;
3. convertê-lo para o nome da classe usando `id2label`;
4. exibir a classe prevista.


In [ ]:
# Escreva sua solução aqui.

exercise_text = "o suporte foi excelente"

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q11.hint()` e `q11.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q11 = TILExercise(
    hint_text=(
        "Use o `tokenizer` com `return_tensors='pt'`. "
        "Execute `model(**inputs)` dentro de `torch.no_grad()`, use `argmax(dim=-1)` nos logits e consulte `id2label`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "inputs = tokenizer(exercise_text, return_tensors='pt', truncation=True, max_length=64)\n"
        "model.eval()\n"
        "with torch.no_grad():\n"
        "    outputs = model(**inputs)\n"
        "predicted_id = outputs.logits.argmax(dim=-1).item()\n"
        "print('Classe prevista:', id2label[predicted_id])\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q11.hint() ou q11.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q11.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q11.solution()


## 15. Reprodutibilidade

- linguagem: Python;
- bibliotecas principais: `torch`, `transformers`, `scikit-learn`, `numpy`;
- modelo: `distilbert-base-multilingual-cased`;
- recurso: Kaggle Model `goddiao/distilbert-base-multilingual-cased`, PyTorch/default, Version 1;
- tarefa: classificação de sequência;
- epochs: 1;
- learning rate: `2e-5`;
- batch size: 4;
- seed: 42;
- internet: **desabilitada**;
- carregamento do modelo: caminho local Kaggle + `local_files_only=True`;
- dependência externa em tempo de execução: nenhuma, desde que o modelo esteja anexado ao notebook.


## 16. Resumo

Nesta aula, você aprendeu que:

- modelos pré-treinados podem ser adaptados por fine-tuning;
- tokenizer e modelo formam um pipeline integrado;
- BERT-like models usam representações contextuais;
- classification heads adaptam o encoder a classes específicas;
- fine-tuning é uma forma de transfer learning;
- modelos Transformer aumentam capacidade, custo e complexidade;
- baselines clássicos continuam essenciais para comparação.

### Ideia principal

```text
Não treinamos linguagem do zero.
Reaproveitamos conhecimento pré-treinado
e o adaptamos à tarefa que realmente importa.
```

**Fim da Aula 11.**
